# Querying On-Premise Apache Iceberg from Snowflake

This notebook demonstrates every query pattern available once the Iceberg tunnel is running.

```
Snowflake Notebook / UDF
      │
      ▼  (HTTP, internal SPCS network)
iceberg-agent  (port 8090)
      │  proxies Iceberg REST + S3, rewrites minio:9000 → /s3
      ▼  (TCP via tunnel-sidecar)
On-Premise:
  Lakekeeper REST  :8181
  S3-compatible    :9000
```

**Prerequisites:**
- `websocket_multi_db_service` is running in SPCS (`SHOW SERVICES LIKE 'websocket_multi_db_service'`)
- On-premise agent is connected and shows `📡 Listening for forwarding requests...`
- UDFs and stored procedures from `setup.sql` have been created
- Demo tables seeded via `pixi run seed`

---

### Query patterns — sovereignty spectrum

| Option | Method | Data in Snowflake? | Governance? |
|--------|--------|-------------------|-------------|
| A | Direct DuckDB | Never | No |
| B | PyIceberg | Never | No |
| C | `query_iceberg` UDF | No (VARIANT in query) | No |
| D | `query_iceberg_to_table` UDF | Yes (explicit) | ✅ Full Horizon |
| E | `sync_and_query` stored proc | Yes (explicit) | ✅ Full Horizon |

## Configuration

Set your Snowflake database and schema below — these determine the internal SPCS DNS name
for the iceberg-agent.

> **SPCS internal DNS format:** `<service-name>.<schema>.<database>.svc.spcs.internal`  
> Snowflake lowercases the names and replaces underscores with hyphens.

In [ ]:
# ── Edit these to match your Snowflake setup ───────────────────────────────────
SF_DATABASE = "sovereignty_demo"   # the database where the service lives
SF_SCHEMA   = "opt1_tunnel"        # the schema where the service lives
# ───────────────────────────────────────────────────────────────────────────────

SERVICE_NAME = "tunnel-service"

# Internal SPCS URL — reachable from any notebook or UDF in the same account
AGENT_URL = (
    f"http://{SERVICE_NAME}"
    f".{SF_SCHEMA.lower().replace('_', '-')}"
    f".{SF_DATABASE.lower().replace('_', '-')}"
    f".svc.spcs.internal:8090"
)

print(f"Agent URL: {AGENT_URL}")

## Install packages

Only needed the first time, or if running outside a Snowflake Notebook environment.

In [ ]:
# Uncomment if packages are not already available in your environment
# !pip install duckdb pyiceberg s3fs

---
## Option A — Direct DuckDB via iceberg-agent proxy (ephemeral)

DuckDB connects directly to the iceberg-agent's internal SPCS URL. The agent:
- Proxies Iceberg REST catalog requests (`/catalog/`)
- Proxies all MinIO S3 object reads (`/s3/`), rewriting `minio:9000` references transparently

Results live in DuckDB's in-memory engine inside the notebook's Python runtime.
Nothing is ever written to Snowflake storage.

**Sovereignty:** maximum — Snowflake has no copy of the data, ever. No result set cache applies.

In [ ]:
import duckdb

con = duckdb.connect(":memory:")
con.execute("INSTALL iceberg; LOAD iceberg;")
con.execute("INSTALL httpfs;  LOAD httpfs;")

# S3 secret points at the agent's /s3 proxy — agent rewrites minio:9000 internally
con.execute(f"""
    CREATE SECRET iceberg_s3 (
        TYPE      S3,
        KEY_ID    'admin',
        SECRET    'password',
        ENDPOINT  '{AGENT_URL.replace("http://", "")}',
        USE_SSL   false,
        URL_STYLE 'path',
        REGION    'us-east-1'
    );
""")
con.execute("CREATE SECRET iceberg_rest (TYPE ICEBERG, TOKEN 'dummy');")

# Attach via the agent's /catalog proxy
con.execute(f"""
    ATTACH 'demo' AS demo (
        TYPE     ICEBERG,
        ENDPOINT '{AGENT_URL}/catalog/',
        SECRET   iceberg_rest,
        ACCESS_DELEGATION_MODE 'none'
    );
""")

df = con.execute("SELECT * FROM demo.demo.sales ORDER BY sale_id").fetchdf()
print(df)

### Option 1 – join on-prem sensitive data with Snowflake data, in memory only
`EMPLOYEES_PUBLIC` lives in Snowflake; `employees_sensitive` never leaves on-prem storage except as transient bytes through the tunnel into this DuckDB process.

In [ ]:
from snowflake.snowpark.context import get_active_session
public_df = get_active_session().table('SOVEREIGNTY_DEMO.OPT1_TUNNEL.EMPLOYEES_PUBLIC').to_pandas()
con.register('employees_public', public_df)
con.execute('''
    SELECT p.DEPARTMENT, COUNT(*) AS n, ROUND(AVG(s.salary)) AS avg_salary
    FROM employees_public p
    JOIN demo.demo.employees_sensitive s ON s.employee_id = p.EMPLOYEE_ID
    GROUP BY 1 ORDER BY 1
''').fetchdf()

In [ ]:
# Aggregation — stays entirely in DuckDB memory
agg = con.execute("""
    SELECT region, SUM(amount) AS total_sales, COUNT(*) AS num_sales
    FROM demo.demo.sales
    GROUP BY region
    ORDER BY total_sales DESC
""").fetchdf()
print(agg)

---
## Option B — PyIceberg (ephemeral, Pythonic)

PyIceberg reads the Iceberg catalog and data files directly through the agent proxy.
Results are returned as a PyArrow table or pandas DataFrame.

Same sovereignty guarantee as Option A: nothing persists in Snowflake.

In [ ]:
from pyiceberg.catalog.rest import RestCatalog

catalog = RestCatalog(
    name="demo",
    uri=f"{AGENT_URL}/catalog/",
    warehouse="demo",
)

# List available tables
print("Namespaces:", catalog.list_namespaces())
print("Tables:    ", catalog.list_tables("demo"))

# Scan sales table into a pandas DataFrame
table = catalog.load_table("demo.sales")
df = table.scan().to_pandas()
print(df)

In [ ]:
# Predicate pushdown — only fetch rows where amount > 100
from pyiceberg.expressions import GreaterThan

df_filtered = table.scan(
    row_filter=GreaterThan("amount", 100)
).to_pandas()
print(df_filtered)

---
## Option C — Snowflake UDF (`query_iceberg`)

The `query_iceberg` UDF executes a query via DuckDB inside the `iceberg-agent` container
and returns results as a Snowflake `VARIANT`. Nothing is written to Snowflake storage.

**Sovereignty:** ephemeral — result exists only for the duration of the call.  
**Note:** Container UDFs are scalar today (`MAX_BATCH_ROWS = 1`). Best for bounded queries.

In [ ]:
import json
from snowflake.snowpark.context import get_active_session

session = get_active_session()

result = session.sql("""
    SELECT query_iceberg(
        'SELECT * FROM demo.demo.sales ORDER BY sale_id',
        'demo'
    ) AS iceberg_data
""").collect()

rows = json.loads(result[0][0])
print(f"Rows returned: {len(rows)}")
for row in rows:
    print(row)

---
## Option D — Table-sync UDF (`query_iceberg_to_table`) — persisted, governed

The `query_iceberg_to_table` UDF executes the query via DuckDB and writes the result
directly into a native Snowflake table using the SPCS OAuth token. The UDF returns a
status object; the rows land in `SALES_FROM_ICEBERG`.

**Sovereignty:** data is explicitly moved into Snowflake — this is an intentional choice.  
**Benefit:** full Snowflake Horizon governance applies (masking policies, RBAC, lineage).

In [ ]:
import json
from snowflake.snowpark.context import get_active_session

session = get_active_session()

sync_result = session.sql("""
    SELECT query_iceberg_to_table(
        'SELECT * FROM demo.demo.sales ORDER BY sale_id',
        'demo',
        'SALES_FROM_ICEBERG'
    )
""").collect()

print(json.loads(sync_result[0][0]))

In [ ]:
# Query the synced table natively — it's a regular Snowflake table now
df = session.sql("SELECT * FROM SALES_FROM_ICEBERG ORDER BY SALE_ID").to_pandas()
print(df)

In [ ]:
# Apply a column masking policy to the synced table
# Analysts without SYSADMIN or DOCKERTEST role see -1 instead of real amounts
session.sql("""
    CREATE OR REPLACE MASKING POLICY amount_mask AS (val NUMBER)
      RETURNS NUMBER ->
        CASE
          WHEN CURRENT_ROLE() IN ('ACCOUNTADMIN', 'SYSADMIN', 'DOCKERTEST') THEN val
          ELSE -1
        END
""").collect()

session.sql("""
    ALTER TABLE SALES_FROM_ICEBERG
      MODIFY COLUMN AMOUNT
      SET MASKING POLICY amount_mask
""").collect()

print("Masking policy applied to SALES_FROM_ICEBERG.AMOUNT")

---
## Option E — Stored procedure (`sync_and_query`) — sync + return in one call

`sync_and_query` chains the table-sync UDF with an immediate `SELECT`, so callers
get live rows back in a single round-trip. Useful for dashboards and BI tools.

Wrap in a Snowflake Task to schedule automatic syncs:
```sql
CREATE TASK sync_sales_daily
  WAREHOUSE = <your-warehouse>
  SCHEDULE  = 'USING CRON 0 6 * * * UTC'
AS
  CALL sync_and_query('SELECT * FROM demo.demo.sales', 'demo', 'SALES_FROM_ICEBERG');
```

In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()

df = session.sql("""
    CALL sync_and_query(
        'SELECT * FROM demo.demo.sales ORDER BY sale_id',
        'demo',
        'SALES_FROM_ICEBERG'
    )
""").to_pandas()

print(df)

---
## Joining On-Premise Iceberg with a Native Snowflake Table

Two approaches shown:

- **Approach 1 (DuckDB in-memory):** pull Snowflake data into a pandas DataFrame,
  register it in DuckDB alongside the Iceberg table, and join in DuckDB memory.
  Neither dataset is persisted anywhere — fully ephemeral.

- **Approach 2 (Snowpark SQL):** sync Iceberg data into a Snowflake table first,
  then join natively. Data persists; full Snowflake governance applies.

In [ ]:
# ── Approach 1: DuckDB in-memory join (fully ephemeral) ───────────────────────
# Pull the Snowflake dimension table into pandas, register it in DuckDB,
# and join against the live Iceberg table — nothing persists anywhere.

import duckdb
from snowflake.snowpark.context import get_active_session

session = get_active_session()

# Ensure a native customer dimension table exists
session.sql("""
    CREATE TABLE IF NOT EXISTS SF_CUSTOMERS (
        customer_id   INTEGER,
        customer_name VARCHAR,
        country       VARCHAR,
        tier          VARCHAR
    )
""").collect()
session.sql("""
    INSERT INTO SF_CUSTOMERS
    SELECT * FROM VALUES
        (1, 'Acme Corp',    'USA',     'Gold'),
        (2, 'Globex',       'UK',      'Silver'),
        (3, 'Initech',      'Germany', 'Gold'),
        (4, 'Umbrella Inc', 'France',  'Bronze'),
        (5, 'Hooli',        'USA',     'Silver')
    t(customer_id, customer_name, country, tier)
""").collect()

# Load Snowflake dimension into pandas
sf_customers = session.table("SF_CUSTOMERS").to_pandas()
sf_customers.columns = sf_customers.columns.str.lower()

# Set up DuckDB with Iceberg (reuse con from Option A if still in scope,
# or re-initialise here)
con = duckdb.connect(":memory:")
con.execute("INSTALL iceberg; LOAD iceberg; INSTALL httpfs; LOAD httpfs;")
con.execute(f"""
    CREATE SECRET iceberg_s3 (
        TYPE S3, KEY_ID 'admin', SECRET 'password',
        ENDPOINT '{AGENT_URL.replace("http://", "")}',
        USE_SSL false, URL_STYLE 'path'
    );
""")
con.execute("CREATE SECRET iceberg_rest (TYPE ICEBERG, TOKEN 'dummy');")
con.execute(f"""
    ATTACH 'demo' AS demo (
        TYPE ICEBERG,
        ENDPOINT '{AGENT_URL}/catalog/',
        SECRET iceberg_rest,
        ACCESS_DELEGATION_MODE 'none'
    );
""")

# Register the Snowflake DataFrame in DuckDB
con.register("sf_customers", sf_customers)

# Join on-premise Iceberg sales with Snowflake customers — entirely in DuckDB memory
result = con.execute("""
    SELECT
        i.sale_id, i.product, i.amount, i.sale_date, i.region,
        s.customer_name, s.tier
    FROM demo.demo.sales  i
    JOIN sf_customers     s ON i.sale_id = s.customer_id
    ORDER BY i.sale_id
""").fetchdf()

print(result)

In [ ]:
# ── Approach 2: Snowpark SQL join (requires SALES_FROM_ICEBERG from Option D/E) ─
from snowflake.snowpark.context import get_active_session

session = get_active_session()

joined = session.sql("""
    SELECT
        s.sale_id,
        s.product,
        s.amount,
        s.region,
        c.customer_name,
        c.tier
    FROM SALES_FROM_ICEBERG s
    LEFT JOIN SF_CUSTOMERS  c ON s.sale_id = c.customer_id
    ORDER BY s.sale_id
""").to_pandas()

print(joined)